Import primary library

In [132]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display
import json

np.set_printoptions(
    precision=4,      # 4 chữ số thập phân
    suppress=True,    # không dùng dạng 1.23e-05
    linewidth=200     # tránh xuống dòng quá sớm
)

In [133]:
import os
import kagglehub
import shutil
from sklearn.feature_extraction.text import CountVectorizer

Load Data

In [134]:
path = kagglehub.dataset_download(
    "arashnic/mind-news-dataset"
)

print("Dataset downloaded to: ", path)

Dataset downloaded to:  C:\Users\tonmi\.cache\kagglehub\datasets\arashnic\mind-news-dataset\versions\2


Copy data to working dir

In [135]:
source = os.path.join(path, "MINDsmall_train")

destination = r"D:\CDNC\MIND-research\data\raw"

os.makedirs(destination, exist_ok=True)

files = [
    "news.tsv",
    "behaviors.tsv",
    "entity_embedding.vec",
    "relation_embedding.vec",
]

for file in files:
    shutil.copy2(
        os.path.join(source, file),
        os.path.join(destination, file)
    )

print("Done")



Done


Get split-dataset - 300 line - News_ID + Category + News title

In [136]:
news_path = os.path.join(destination, "news.tsv")

if not os.path.exists(news_path):
    f_news_small = open(news_path, "x", encoding="utf-8")


columns = [
    "News_ID",
    "Category",
    "SubCategory",
    "Title",
    "Abstract",
    "URL",
    "Title_Entities",
    "Abstract_Entities"
]

news = pd.read_csv(
    news_path,
    sep="\t",
    names=columns,
    
)

news = news[["News_ID", "Category", "Title"]]

sample = news.sample(
    n=300,
    random_state=42
).reset_index(drop=True)

sample = news.head(300)
sample_dir = r"D:\CDNC\MIND-research\data\sample"

sample.to_csv(
    os.path.join(sample_dir, "news.csv"),
    index=False
)

print(sample.head())
print("Sample shape: ", sample.shape)

  News_ID   Category                                              Title
0  N55528  lifestyle  The Brands Queen Elizabeth, Prince Charles, an...
1  N19639     health                      50 Worst Habits For Belly Fat
2  N61837       news  The Cost of Trump's Aid Freeze in the Trenches...
3  N53526     health  I Was An NBA Wife. Here's How It Affected My M...
4  N38324     health  How to Get Rid of Skin Tags, According to a De...
Sample shape:  (300, 3)


In [137]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

class Models:
    def __init__(self):
        self.sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

        self.svectorizer = CountVectorizer(
                stop_words="english",
            )

        self.topic_model = BERTopic(
            calculate_probabilities=True, # Important to set this to True for probability calculations
            verbose=True,
            vectorizer_model=self.svectorizer
        )

models = Models()

class VectorContext:
    def __init__(self):
        self.title_list = None
        self.semantic_vector_list = None
        self.probabilities_list = None
        self.topics_vector_list = None

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Vector Factory

`VectorFactory` được sử dụng để tạo đối tượng biểu diễn vector tương ứng với từng mô hình thông qua một giao diện thống nhất. Thay vì khởi tạo trực tiếp từng lớp, người dùng chỉ cần chỉ định loại mô hình (`sentence` hoặc `bertopic`), Factory sẽ trả về đối tượng phù hợp.

Thiết kế này giúp:
- Tách biệt logic khởi tạo khỏi logic xử lý.
- Dễ dàng thay thế hoặc mở rộng sang các mô hình mới mà không ảnh hưởng đến mã nguồn hiện có.
- Tăng khả năng bảo trì và tái sử dụng mã nguồn.

Vector Interface

In [138]:
from abc import ABC, abstractmethod

class Vector(ABC):
    @abstractmethod
    def get_vector(self):
        pass
    
    @abstractmethod
    def overview(self):
        pass

    @abstractmethod
    def summary(self):
        pass

Semantic vector

In [139]:
class SentenceVector(Vector):
    def __init__(self, model):
        self.model = model
        self.semantic_vector = None

    def get_vector(self, title_list):
        self.title_list = title_list
        
        titles = [news["title"] for news in title_list]

        vectors = self.model.encode(
            titles,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        
        self.semantic_vector = {
            news["news_id"]: vector for news, vector in zip(title_list, vectors)
        }
        return self.semantic_vector
    
    def overview(self):
        print("=" * 60)
        print("Sentence Embedding Overview")
        print(f"[Sentence is: {list(self.semantic_vector.keys())[0]}]")
        print("=" * 60)

        print(f"Documents : {len(self.semantic_vector)}")
        print(f"Dimension : {self.semantic_vector[list(self.semantic_vector.keys())[0]].shape[0]}")
        print(f"Shape     : {self.semantic_vector[list(self.semantic_vector.keys())[0]].shape}")

        print("\nFirst vector (first 10 values):")
        print(self.semantic_vector[list(self.semantic_vector.keys())[0]][:10], "...")

    def summary(self, sample_index=0):

        sample = self.title_list[sample_index]

        news_id = sample["news_id"]
        title = sample["title"]

        vector = self.semantic_vector[news_id]

        metrics = pd.DataFrame({
            "Metric": [
                "Embedding Model",
                "Number of Documents",
                "Embedding Dimension",
                "Output Shape"
            ],
            "Value": [
                self.model.__class__.__name__,
                len(self.semantic_vector),
                vector.shape[0],
                str(vector.shape)
            ]
        })

        vector_preview = ", ".join(
            f"{x:.4f}" for x in vector[:10]
        ) + ", ..."

        example = pd.DataFrame({
            "News ID": [news_id],
            "Sample Title": [title],
            "Embedding (first 10 dims)": [
                f"[{vector_preview}]"
            ]
        })

        return metrics, example


Topic vector

In [140]:
class BERTopicVector(Vector):
    def __init__(self, model):
        self.model = model

        self.notice = "BERTopic depends on the sentence transformer model." \
        " Please ensure that the sentence transformer model is trained before using BERTopic."

        self.probabilities = None
        self.topics_vector = None

    def get_vector(self, title_list, semantic_vector=None):
        self.title_list = title_list
        
        titles = [news["title"] for news in title_list]

        embeddings = np.array([
            semantic_vector[news["news_id"]]
            for news in title_list
        ])
        
        if semantic_vector is None:
            print(self.notice)
            return
        
        topics, probabilities = self.model.fit_transform(
                                    titles,
                                    embeddings,
                                )
        
        self.topic_vector = {
            news["news_id"]: {
                "topic": topic,
                "probability": probability
            }
            for news, topic, probability in zip(
                title_list,
                topics,
                probabilities
            )
        }

        return self.topic_vector
    
    def overview(self):

        print("=" * 60)
        print("BERTopic Overview")
        print("=" * 60)

        print(f"Number of documents : {len(self.title_list)}")
        print(f"Number of topics    : {len(set(v['topic'] for v in self.topic_vector.values()) - {-1})}")

        print("\nTopic distribution:")
        print(self.model.get_topic_info()[["Topic", "Count"]])

        print("\nFirst 5 documents:")

        for sample in self.title_list[:5]:

            news_id = sample["news_id"]
            title = sample["title"]

            topic = self.topic_vector[news_id]["topic"]

            print(f"{news_id}")
            print(f"Title : {title}")
            print(f"Topic : {topic}")
            print("-" * 40)

        first_probability = next(iter(self.topic_vector.values()))["probability"]

        print("\nProbability shape:")
        print(first_probability.shape)

        print("\nFirst 5 probability vectors:")

        for sample in self.title_list[:5]:

            news_id = sample["news_id"]

            probability = self.topic_vector[news_id]["probability"]

            preview = ", ".join(
                f"{p:.4f}" for p in probability[:10]
            )

            print(f"{news_id} -> [{preview}, ...]")

    
    def summary(self, sample_index=0):

        # =========================
        # Metrics
        # =========================

        topics = [
            value["topic"]
            for value in self.topic_vector.values()
        ]

        first_probability = next(iter(self.topic_vector.values()))["probability"]

        metrics_df = pd.DataFrame({
            "Metric": [
                "Topic Model",
                "Number of Documents",
                "Number of Topics",
                "Number of Outliers",
                "Probability Shape"
            ],
            "Value": [
                self.model.__class__.__name__,
                len(self.title_list),
                len(set(topics) - {-1}),
                np.sum(np.array(topics) == -1),
                str(first_probability.shape)
            ]
        })

        # =========================
        # Topic Information
        # =========================

        topic_df = self.model.get_topic_info()[["Topic", "Count"]].copy()

        keywords = []

        for topic in topic_df["Topic"]:

            if topic == -1:
                keywords.append("Outlier")
            else:
                words = [
                    word
                    for word, _ in self.model.get_topic(topic)[:5]
                ]
                keywords.append(", ".join(words))

        topic_df["Top Keywords"] = keywords

        # =========================
        # Sample
        # =========================

        sample = self.title_list[sample_index]

        news_id = sample["news_id"]
        title = sample["title"]

        topic_info = self.topic_vector[news_id]

        probs = ", ".join(
            f"{p:.4f}"
            for p in topic_info["probability"]
        )

        sample_df = pd.DataFrame({
            "News ID": [news_id],
            "Sample Title": [title],
            "Assigned Topic": [
                topic_info["topic"]
            ],
            "Probability Distribution": [
                f"[{probs}]"
            ]
        })

        return metrics_df, topic_df, sample_df

Title list

In [141]:
model_context = VectorContext()
to_dict = lambda news: {"news_id": news["News_ID"], "title": news["Title"]}
model_context.title_list = [to_dict(news) for _, news in sample.iterrows()]

print(model_context.title_list)


[{'news_id': 'N55528', 'title': 'The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By'}, {'news_id': 'N19639', 'title': '50 Worst Habits For Belly Fat'}, {'news_id': 'N61837', 'title': "The Cost of Trump's Aid Freeze in the Trenches of Ukraine's War"}, {'news_id': 'N53526', 'title': "I Was An NBA Wife. Here's How It Affected My Mental Health."}, {'news_id': 'N38324', 'title': 'How to Get Rid of Skin Tags, According to a Dermatologist'}, {'news_id': 'N2073', 'title': 'Should NFL be able to fine players for criticizing officiating?'}, {'news_id': 'N49186', 'title': "It's been Orlando's hottest October ever so far, but cooler temperatures on the way"}, {'news_id': 'N59295', 'title': 'Chile: Three die in supermarket fire amid protests'}, {'news_id': 'N24510', 'title': 'Best PS5 games: top PlayStation 5 titles to look forward to'}, {'news_id': 'N39237', 'title': 'How to report weather-related closings, delays'}, {'news_id': 'N9721', 'title': '50 Foods You Should Never Eat,

Semantic vector

In [142]:
sentence_vector= SentenceVector(models.sentence_model)

model_context.semantic_vector_list = sentence_vector.get_vector(model_context.title_list)
# print(model_context.semantic_vector_list)
sentence_vector.overview()

metric, example = sentence_vector.summary()

print("\nSummary of Sentence Embedding:")
display(metric)
print("\nExample of Sentence Embedding:")
display(example)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Sentence Embedding Overview
[Sentence is: N55528]
Documents : 300
Dimension : 384
Shape     : (384,)

First vector (first 10 values):
[-0.0093  0.0424  0.059   0.0121  0.0334  0.0196  0.0274 -0.0615 -0.0362 -0.039 ] ...

Summary of Sentence Embedding:


,Metric,Value
0,Embedding Model,SentenceTransformer
1,Number of Documents,300
2,Embedding Dimension,384
3,Output Shape,"(384,)"



Example of Sentence Embedding:


,News ID,Sample Title,Embedding (first 10 dims)
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an...","[-0.0093, 0.0424, 0.0590, 0.0121, 0.0334, 0.01..."


In [143]:
bertopic_vector = BERTopicVector(models.topic_model)

topics_vector = bertopic_vector.get_vector(model_context.title_list, model_context.semantic_vector_list)
model_context.topics_vector_list = topics_vector
bertopic_vector.overview()

metric, topic, example = bertopic_vector.summary()
print("\nSummary of BERTopic:")
display(metric)
print("\nTopic Information:")
display(topic)
print("\nExample of BERTopic:")
display(example)

2026-07-17 22:14:08,819 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-17 22:14:08,993 - BERTopic - Dimensionality - Completed ✓
2026-07-17 22:14:08,995 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-17 22:14:09,016 - BERTopic - Cluster - Completed ✓
2026-07-17 22:14:09,019 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-17 22:14:09,037 - BERTopic - Representation - Completed ✓


BERTopic Overview
Number of documents : 300
Number of topics    : 6

Topic distribution:
   Topic  Count
0     -1     36
1      0    116
2      1     41
3      2     39
4      3     29
5      4     23
6      5     16

First 5 documents:
N55528
Title : The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By
Topic : 1
----------------------------------------
N19639
Title : 50 Worst Habits For Belly Fat
Topic : 0
----------------------------------------
N61837
Title : The Cost of Trump's Aid Freeze in the Trenches of Ukraine's War
Topic : 5
----------------------------------------
N53526
Title : I Was An NBA Wife. Here's How It Affected My Mental Health.
Topic : 0
----------------------------------------
N38324
Title : How to Get Rid of Skin Tags, According to a Dermatologist
Topic : 0
----------------------------------------

Probability shape:
(6,)

First 5 probability vectors:
N55528 -> [0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, ...]
N19639 -> [1.0000, 0.0000, 0.00

,Metric,Value
0,Topic Model,BERTopic
1,Number of Documents,300
2,Number of Topics,6
3,Number of Outliers,36
4,Probability Shape,"(6,)"



Topic Information:


,Topic,Count,Top Keywords
0,-1,36,Outlier
1,0,116,"things, make, 2019, 50, recipes"
2,1,41,"royal, kate, queen, prince, star"
3,2,39,"nfl, football, week, season, patriots"
4,3,29,"tv, vegas, ford, chevy, 2021"
5,4,23,"police, killed, man, causes, maryland"
6,5,16,"2020, house, clinton, trumps, election"



Example of BERTopic:


,News ID,Sample Title,Assigned Topic,Probability Distribution
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an...",1,"[0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000]"


Represented vector

In [171]:
class RepresentedVector(Vector):
    def __init__(self, title_list, sentence_dict, bertopic_dict):
        self.title_list = title_list
        self.sentence_dict = sentence_dict
        self.bertopic_dict = bertopic_dict
        self.represented_vector = {}
      
    def get_vector(self):
        self.represented_vector = {
            news["news_id"]: {
                "title": news["title"],
                "semantic": self.sentence_dict[news["news_id"]],
                "topic": self.bertopic_dict[news["news_id"]]["topic"],
                "topic_distribution": self.bertopic_dict[news["news_id"]]["probability"]
            }
            for news in self.title_list
        }

        return self.represented_vector

    def overview(self):
        print("=" * 80)
        print("Represented Vector Overview")
        print("=" * 80)

        print(f"Documents           : {len(self.represented_vector)}")

        first_news_id = next(iter(self.represented_vector))

        sample = self.represented_vector[first_news_id]

        print(f"Semantic Dimension  : {len(sample['semantic'])}")
        print(f"Topic Distribution  : {len(sample['topic_distribution'])}")
        print(f"Stored Fields       : {list(sample.keys())}")

        print("=" * 80)
        
    def preview_vector(vector, preview_dims=4):
        vector = [round(float(x), 4) for x in vector]

        if len(vector) <= preview_dims * 2:
            return vector

        return (
            vector[:preview_dims]
            + ["..."]
            + vector[-preview_dims:]
        )

    def summary(self, sample_index=0, preview_dims=4):

        news = self.title_list[sample_index]
        news_id = news["news_id"]

        represented = self.represented_vector[news_id]

        semantic = represented["semantic"]
        probability = represented["topic_distribution"]

        def preview(vector):
            vector = [round(float(x), 4) for x in vector]

            if len(vector) <= preview_dims * 2:
                return vector

            return (
                vector[:preview_dims]
                + ["..."]
                + vector[-preview_dims:]
            )

        semantic_preview = preview(semantic)
        probability_preview = preview(probability)

        summary_df = pd.DataFrame({
            "Field": [
                "News ID",
                "Title",
                "Assigned Topic",
                "Semantic Dimension",
                "Topic Distribution Dimension"
            ],
            "Value": [
                news_id,
                represented["title"],
                represented["topic"],
                len(semantic),
                len(probability)
            ]
        })

        represented_preview = {
            news_id: {
                "title": represented["title"],
                "semantic": semantic_preview,
                "topic": represented["topic"],
                "topic_distribution": probability_preview
            }
        }

        return summary_df, represented_preview

In [172]:
represented_vector = RepresentedVector(model_context.title_list, model_context.semantic_vector_list, model_context.topics_vector_list)

# print(model_context.semantic_vector_list)
# print(model_context.semantic_vector_list)
# print(model_context.topics_vector_list)

represented_vector.get_vector()
represented_vector.overview()
represented_vector.summary(sample_index=0, preview_dims=4)

Represented Vector Overview
Documents           : 300
Semantic Dimension  : 384
Topic Distribution  : 6
Stored Fields       : ['title', 'semantic', 'topic', 'topic_distribution']


(                          Field  \
 0                       News ID   
 1                         Title   
 2                Assigned Topic   
 3            Semantic Dimension   
 4  Topic Distribution Dimension   
 
                                                Value  
 0                                             N55528  
 1  The Brands Queen Elizabeth, Prince Charles, an...  
 2                                                  1  
 3                                                384  
 4                                                  6  ,
 {'N55528': {'title': 'The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By',
   'semantic': [-0.0093,
    0.0424,
    0.059,
    0.0121,
    '...',
    0.0517,
    -0.1316,
    0.0508,
    -0.0301],
   'topic': 1,
   'topic_distribution': [0.0, 1.0, 0.0, 0.0, 0.0, 0.0]}})

#   User Representation Vector

##  Prepare sample data User Representation

In [ ]:
behaviours_path = os.path.join(destination, "behaviors.tsv")

news = pd.read_csv(
    news_path,
    sep="\t",
    names=[
        "news_id",
        "category",
        "subcategory",
        "title",
        "abstract",
        "url",
        "title_entities",
        "abstract_entities"
    ]
)

news_dict = news.set_index("news_id")["title"].to_dict()

columns_behaviours = [
    "user_id",
    "time",
    "history",
    "impressions"
]

behaviours = pd.read_csv(
    behaviours_path,
    sep="\t",
    names=columns_behaviours,
    
)

behaviours = behaviours[["user_id", "history"]]
behaviours = behaviours.dropna(subset=["history"])

behaviours["titles"] = behaviours["history"].apply(
    lambda history: [
        news_dict[news_id]
        for news_id in history.split()
        if news_id in news_dict
    ]
)

sample = behaviours.sample(
    n=10,
    random_state=42
).reset_index(drop=True)

sample_dir = r"D:\CDNC\MIND-research\data\sample"

sample.to_csv(
    os.path.join(sample_dir, "behaviours.csv"),
    index=False
)

print(sample.head())
print("Sample shape: ", sample.shape)

## Get Represented Vector of 1 user

In [40]:
repres = represented_vector.get_vector()
print(len(repres))

300
